In [6]:
# import
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src/data"

sys.path.append(str(SRC_DIR))

import torch
from vocab import Vocabulary
from vi_tokenizer import clean_text_vi, tokenize_vi
from en_tokenizer import EnglishBPETokenizer
from dataset import TranslationDataset, CollateBatch, get_dataloader

In [7]:
# test Vietnamese tokenizer
assert clean_text_vi("Xin Chào!!!") == "xin chào ! ! !"

tokens = tokenize_vi("Tôi yêu tiếng Việt.")
assert tokens[-1] == "."
assert "tôi" in tokens
print("Vietnamese tokenizer OK:", tokens)

Vietnamese tokenizer OK: ['tôi', 'yêu', 'tiếng', 'việt', '.']


In [ ]:
# test English BPE tokenizer
tok_en = EnglishBPETokenizer(num_merges=10)

cleaned = tok_en.clean_text_en("I'm 15 years old!")
assert "i am" in cleaned
assert "<num>" in cleaned
assert "!" in cleaned

tok_en.train(["I am happy.", "I am very happy.", "You are happy."])
encoded = tok_en.encode("I am happy.")
assert isinstance(encoded, list)
assert all(isinstance(x, str) for x in encoded)

print("English BPE tokenizer OK:", encoded)

English BPE tokenizer OK: ['265', '262']


In [9]:
# test Vocabulary
vocab = Vocabulary(freq_threshold=2)

sentences = [
    ["i", "am", "happy"],
    ["you", "are", "happy"],
    ["i", "am", "student"],
]

vocab.build_vocabulary(sentences)

assert vocab.stoi["<pad>"] == 0
assert vocab.stoi["<sos>"] == 1
assert vocab.stoi["<eos>"] == 2
assert vocab.stoi["<unk>"] == 3

assert "i" in vocab.stoi
assert "am" in vocab.stoi
assert "happy" in vocab.stoi
assert "student" not in vocab.stoi

nums = vocab.numericalize(["<sos>", "i", "unknown_word", "<eos>"])
assert nums[0] == 1
assert nums[-1] == 2
assert nums[2] == 3

print("Vocabulary OK:", nums)

Vocabulary OK: [1, 5, 3, 2]


In [10]:
# test TranslationDataset
src_texts = ["I am happy.", "You are student."]
trg_texts = ["Tôi vui.", "Bạn là học sinh."]

tok_en = EnglishBPETokenizer(num_merges=20)
tok_en.train(src_texts)

tokenizer_src = tok_en.encode
tokenizer_trg = tokenize_vi

src_tokenized = [
    ["<sos>"] + tokenizer_src(s) + ["<eos>"]
    for s in src_texts
]

trg_tokenized = [
    ["<sos>"] + tokenizer_trg(s) + ["<eos>"]
    for s in trg_texts
]

vocab_src = Vocabulary(freq_threshold=1)
vocab_trg = Vocabulary(freq_threshold=1)

vocab_src.build_vocabulary(src_tokenized)
vocab_trg.build_vocabulary(trg_tokenized)

dataset = TranslationDataset(
    src_texts,
    trg_texts,
    vocab_src,
    vocab_trg,
    tokenizer_src,
    tokenizer_trg
)

assert len(dataset) == 2

src_ids, trg_ids = dataset[0]
assert src_ids[0] == vocab_src.stoi["<sos>"]
assert src_ids[-1] == vocab_src.stoi["<eos>"]
assert trg_ids[0] == vocab_trg.stoi["<sos>"]
assert trg_ids[-1] == vocab_trg.stoi["<eos>"]

print("TranslationDataset OK")
print("src:", src_ids)
print("trg:", trg_ids)

TranslationDataset OK
src: [4, 5, 6]
trg: [4, 5, 6, 7, 8]


In [11]:
# test CollateBatch
batch = [dataset[0], dataset[1]]

collate = CollateBatch(pad_idx=vocab_src.stoi["<pad>"])
en_padded, vi_padded, en_mask, vi_mask = collate(batch)

assert isinstance(en_padded, torch.Tensor)
assert isinstance(vi_padded, torch.Tensor)
assert en_padded.shape[0] == 2
assert vi_padded.shape[0] == 2
assert en_mask.shape == en_padded.shape
assert vi_mask.shape == vi_padded.shape

assert en_mask.dtype == torch.bool
assert vi_mask.dtype == torch.bool

print("CollateBatch OK")
print("en_padded:\n", en_padded)
print("en_mask:\n", en_mask)

CollateBatch OK
en_padded:
 tensor([[ 4,  5,  6,  0,  0,  0,  0,  0],
        [ 4,  7,  8,  9, 10, 11, 12,  6]])
en_mask:
 tensor([[False, False, False,  True,  True,  True,  True,  True],
        [False, False, False, False, False, False, False, False]])


In [12]:
# test DataLoader
loader = get_dataloader(
    src_texts,
    trg_texts,
    vocab_src,
    vocab_trg,
    tokenizer_src,
    tokenizer_trg,
    batch_size=2
)

batch = next(iter(loader))
en_padded, vi_padded, en_mask, vi_mask = batch

assert en_padded.shape[0] == 2
assert vi_padded.shape[0] == 2
assert en_mask.shape == en_padded.shape
assert vi_mask.shape == vi_padded.shape

print("DataLoader OK")
print(en_padded.shape, vi_padded.shape)

DataLoader OK
torch.Size([2, 8]) torch.Size([2, 7])
